In [ ]:
# Install the project dependencies required by this notebook.
%pip install -q pandas numpy duckdb pyarrow scikit-learn xgboost mlflow matplotlib scipy joblib pyyaml

In [ ]:
# Mount Google Drive so Colab can access the private MIMIC-IV files and derived artifacts.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone or update the GitHub repository so the notebook can import the shared project code.
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mbakos95/aki-sentinel.git"
REPO_DIR = Path("/content/aki-sentinel")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

sys.path.insert(0, str(REPO_DIR))

In [ ]:
# Define the private MIMIC-IV and artifact locations used by the pipeline.
from pathlib import Path

MIMIC_ROOT = Path("/content/drive/MyDrive/MIMIC-IV")
HOSP_DIR = MIMIC_ROOT / "hosp"
ICU_DIR = MIMIC_ROOT / "icu"

PRIVATE_ROOT = Path("/content/drive/MyDrive/AKI-Sentinel-Private")
ARTIFACT_DIR = PRIVATE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the cohort and KDIGO label-building functions used in this notebook.
import pandas as pd

from src.cohort import build_eligible_cohort
from src.labels import build_aki_labels

In [ ]:
# Load the eligible ICU cohort from Drive or rebuild it from the raw MIMIC-IV tables.
COHORT_PATH = ARTIFACT_DIR / "eligible_cohort.parquet"

if COHORT_PATH.exists():
    eligible_cohort = pd.read_parquet(COHORT_PATH)
else:
    eligible_cohort = build_eligible_cohort(HOSP_DIR, ICU_DIR, observation_hours=12)
    eligible_cohort.to_parquet(COHORT_PATH, index=False)

eligible_cohort.shape

In [ ]:
# Build creatinine and urine-output KDIGO stages and create the observable new Stage 2+ AKI target.
labels, creatinine_timeline, urine_timeline, kdigo_events = build_aki_labels(
    cohort=eligible_cohort,
    labevents_path=HOSP_DIR / "labevents.csv.gz",
    outputevents_path=ICU_DIR / "outputevents.csv.gz",
    chartevents_path=ICU_DIR / "chartevents.csv.gz",
    target_stage=2,
)

In [ ]:
# Inspect the final label distribution and the number of ICU stays retained for modeling.
print("Labeled ICU stays:", f"{len(labels):,}")
print("Positive cases:", f"{labels['target'].sum():,}")
print("AKI Stage 2+ prevalence:", f"{labels['target'].mean():.3%}")
labels.head()

In [ ]:
# Save the private label table and intermediate KDIGO timelines for later notebooks.
LABEL_DIR = ARTIFACT_DIR / "labels"
LABEL_DIR.mkdir(parents=True, exist_ok=True)

labels.to_parquet(LABEL_DIR / "aki_labels.parquet", index=False)
creatinine_timeline.to_parquet(LABEL_DIR / "creatinine_timeline.parquet", index=False)
urine_timeline.to_parquet(LABEL_DIR / "urine_timeline.parquet", index=False)
kdigo_events.to_parquet(LABEL_DIR / "kdigo_events.parquet", index=False)